In [1]:
import requests
import re
import pandas as pd
from datetime import datetime
from typing import Match
from bs4 import BeautifulSoup

## カタログ読み込み

In [38]:
res = requests.get("https://may.2chan.net/b/futaba.php?mode=cat", cookies={"cxyl": "100x100x100x1x6"})
bs = BeautifulSoup(res.text, "html.parser")

threads = []
for td in bs.find("table", id="cattable").find_all("td"):
    id_match: Match[str] = re.match(r"res/(\d+?)\.htm", td.a.get("href"))
    id = id_match.group(1)
    if td.a.img:
        imageurl = td.a.img.get("src")
    else:
        imageurl = None
    title = td.small.get_text()

    # "()" で括られてるので[1:-1]で省く
    count = int(td.find("font", size="2").get_text()[1:-1])
    threads.append(
        {"id": int(id), "image_url": imageurl, "title": title, "count": count}
    )

df_threads_html = pd.DataFrame.from_dict(threads)

In [ ]:
res = requests.get("https://may.2chan.net/b/futaba.php?mode=json").json()

threads = []
for k in res["res"].keys():
    date = re.sub(r'\(.*?\)', '', res["res"][k]["now"])
    date = re.match(r'^\d{2}/\d{2}/\d{2}\d{2}:\d{2}:\d{2}', date)
    date = datetime.strptime(date.group(), "%y/%m/%d%H:%M:%S")
    threads.append({
        "id": int(k),
        "date": date,
        "comment": res["res"][k]["com"], 
    })

df_threads_json = pd.DataFrame.from_dict(threads)

,id,date,comment
0,1309462386,2025-04-08 20:54:21,鎧武スレ
1,1309462574,2025-04-08 20:54:52,広末涼子さん
2,1309463019,2025-04-08 20:56:05,ＭＯＷスレ<br><br>安い 美味い 最高
3,1309463167,2025-04-08 20:56:31,むーこかわいい&#129362;
4,1309463267,2025-04-08 20:56:50,ｩ ま女良ﾁｬｿにてチンボｼｭｯｼｭの時間がやって参りました！！
...,...,...,...
536,1309541978,2025-04-09 01:59:39,カテジナスレ<br>頭がおかしい人
537,1309541982,2025-04-09 01:59:41,今ならVTuberも普通に語れると思う
538,1309542052,2025-04-09 02:00:12,これもうなくなるんだってな
539,1309542150,2025-04-09 02:00:58,搾精・搾乳していたら


In [46]:
df_threads = pd.merge(
    df_threads_html,
    df_threads_json,
    on="id",
    how="inner",
)

now = datetime.now()
df_threads['ikioi'] = df_threads['count'] / ((now - df_threads['date']).dt.total_seconds() / 3600)

df_threads = df_threads.sort_values("ikioi", ascending=False)

In [47]:
lines = []

for _, row in df_threads.iterrows():
    lines.append(
        f"スレッドID: {row['id']}\n"
        f"スレッドタイトル: {row['title']}\n"
        f"レス数: {row['count']}\n"
        f"スレッド勢い: {row['ikioi']:.2f}\n"
    )

# 全スレッド分のテキストを1つの文字列にまとめる
result_text = "\n".join(lines)

print(result_text)

スレッドID: 1309534441
スレッドタイトル: ジークアクス
レス数: 1000
スレッド勢い: 925.79

スレッドID: 1309531971
スレッドタイトル: ジークアクス
レス数: 1000
スレッド勢い: 788.51

スレッドID: 1309527135
スレッドタイトル: ジークアクス
レス数: 1000
スレッド勢い: 628.54

スレッドID: 1309524230
スレッドタイトル: ジークアクス実況スレ
レス数: 1000
スレッド勢い: 568.58

スレッドID: 1309523929
スレッドタイトル: ジークアクス
レス数: 1000
スレッド勢い: 561.57

スレッドID: 1309515607
スレッドタイトル: ジークアクス
レス数: 1000
スレッド勢い: 416.01

スレッドID: 1309537190
スレッドタイトル: アポカリプスホテル　このあと40分から
レス数: 331
スレッド勢い: 404.40

スレッドID: 1309510878
スレッドタイトル: トランプ「税金だけ0にして逃げることは許されない。貿易赤字解消するまでアメ車を受け入れる必要がある」
レス数: 1000
スレッド勢い: 365.09

スレッドID: 1309505539
スレッドタイトル: ウマ娘スレ
レス数: 955
スレッド勢い: 310.39

スレッドID: 1309503060
スレッドタイトル: 株スレ
レス数: 1000
スレッド勢い: 309.08

スレッドID: 1309537527
スレッドタイトル: ジークアクス
レス数: 239
スレッド勢い: 305.26

スレッドID: 1309523344
スレッドタイトル: 株スレ
レス数: 530
スレッド勢い: 290.56

スレッドID: 1309496356
スレッドタイトル: お薬やってんですかね？シラフでアレならもっとオツムがアレですが
レス数: 994
スレッド勢い: 272.75

スレッドID: 1309489913
スレッドタイトル: ジークアクス　ガンダムジークアクススレ第一話はいよいよ今夜
レス数: 1000
スレッド勢い: 249.54

スレッドID: 1309476245
スレッドタイトル: 株スレ
レス

## スレッド読み込み

In [2]:
res = requests.get("https://may.2chan.net/b/res/1309537527.htm")
bs = BeautifulSoup(res.text, "html.parser")

In [5]:
thread_bs = bs.find("div", class_="thre")
thread = {"posts": []}

def parse_post(post_bs):
    post = {}

    def gettext_strip(x):
        return x.get_text(strip=True)

    post["sod"] = gettext_strip(post_bs.find("a", class_="sod"))
    post["body"] = post_bs.find("blockquote").get_text(separator="<br>", strip=True)

    return post

thread["posts"].append(parse_post(thread_bs))

for i in bs.find_all("table", border=0):
    thread["posts"].append(parse_post(i))

df_thread = pd.DataFrame.from_dict(thread["posts"])

In [6]:
# "+" を 0 に、"そうだねxN" を N に変換
def parse_sod(s):
    if s == "+":
        return 0
    elif s.startswith("そうだねx"):
        return int(s.replace("そうだねx", ""))
    return 0  # 念のため予備処理

# 変換処理
df_thread['sod_num'] = df_thread['sod'].apply(parse_sod)

# 表示用の文字列を作成
lines = []

for _, row in df_thread.iterrows():
    lines.append(
        f"投稿: {row['body']}\n"
        f"そうだね数: {row['sod_num']}"
    )

# まとめた文字列
result_text = "\n\n".join(lines)

print(result_text)

投稿: ジークアクス
そうだね数: 0

投稿: ビギニング30分で収まるのかな？
そうだね数: 1

投稿: >ジークアクス<br>を見た感想
そうだね数: 6

投稿: この世界からゲルググが居なくなった理由がコイツ
そうだね数: 2

投稿: 削除依頼によって隔離されました<br>レイ＝ユイ<br>カヲル＝ゲンドウ<br>マリ＝キョウコ<br>理解できた？
そうだね数: 3

投稿: いま日テレでエヴァやってて笑う<br>狙ったのかな
そうだね数: 0

投稿: マチュの視認性の良さよ
そうだね数: 0

投稿: このくらいだったな
そうだね数: 3

投稿: 見え
そうだね数: 0

投稿: ない
そうだね数: 3

投稿: >見え<br>>ない<br>真下にいたイクサベ君には世界丸見え
そうだね数: 6

投稿: アメリカのAmazonでも配信始まっとるな<br>Amaプラ版のEDクレジットだと各国語のスタッフ出てるから相当な範囲で公開してるっぽい
そうだね数: 0

投稿: EDCBの録画マージンは-1790秒で大丈夫だったようだ<br>早速テロったけど
そうだね数: 0

投稿: >レイ＝ユイ<br>>カヲル＝ゲンドウ<br>>マリ＝キョウコ<br>>理解できた？<br>君はいい加減消えなさい
そうだね数: 13

投稿: スレッドを立てた人によって削除されました<br>庵野的にはテレビ版21話の時点で綾波レイの正体が碇ユイだと明かしたつもりだった<br>↓<br>それをまったく理解されないままレイが大人気になって「え？」「え？」となった<br>↓<br>カヲルの正体がゲンドウな事もカヲルとレイが夫婦だという事も言いにくくなった<br>↓<br>ところがカヲルも大人気になって謎を明かしきらない方が商売しやすい事を学んだ<br>↓<br>そこで新劇ではマリの正体がキョウコ・ツェッペリンだと明かさなかった<br>↓<br>マリはモヨコ夫人だろ！ふざけんな！！と言われまくって庵野とスタジオカラー終了　　←今ここ
そうだね数: 1

投稿: 赤い
そうだね数: 6

投稿: もしもぢが肛門を越えて外へ抜け出していなければ
そうだね数: 0

投稿: >見え
そうだね数: 6

投稿: >No.1309538366<br>まずきみが書き込んでるスレを理解出来るよ